# 03 — TF-IDF Feature Extraction
Adapts the report appendix's unigram and bigram TF-IDF feature analysis. Fit feature extractors on training data during model evaluation to avoid data leakage; the summaries here are exploratory.

In [ ]:
!pip -q install nltk wordcloud seaborn scikit-learn
import re, numpy as np, pandas as pd, nltk, matplotlib.pyplot as plt, seaborn as sns
from google.colab import files
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.sparse import hstack
for r in ["punkt","punkt_tab","stopwords","wordnet","omw-1.4"]: nltk.download(r, quiet=True)

def clean_tweet(tweet):
    if pd.isna(tweet): return ""
    tweet=str(tweet)
    tweet=re.sub(r"https?://\S+|www\.\S+", " url ", tweet, flags=re.I)
    tweet=re.sub(r"@\w+", " user ", tweet)
    tweet=re.sub(r"#(\w+)", r"\1", tweet)
    tweet=re.sub(r"[^\w\s]", " ", tweet)
    return re.sub(r"\s+", " ", tweet).strip().lower()

def preprocess_text(text, remove_stopwords=True, stem=False, lemmatize=True, custom_stopwords=None):
    tokens=word_tokenize(clean_tweet(text))
    stops=set(stopwords.words("english")) if remove_stopwords else set()
    stops.update(custom_stopwords or [])
    tokens=[w for w in tokens if w not in stops and w.strip()]
    if stem:
        stemmer=PorterStemmer(); tokens=[stemmer.stem(w) for w in tokens]
    if lemmatize:
        lemma=WordNetLemmatizer(); tokens=[lemma.lemmatize(w) for w in tokens]
    return " ".join(tokens)

def load_csv():
    uploaded=files.upload()
    name=next((n for n in uploaded if n.lower().endswith(".csv")),None)
    if not name: raise ValueError("Upload a CSV file.")
    data=pd.read_csv(name)
    if not {"tweet_text","label"}.issubset(data.columns):
        raise ValueError("CSV must contain columns named tweet_text and label. Rename your columns if needed.")
    data=data.dropna(subset=["label"]).copy()
    data["tweet_text"]=data["tweet_text"].fillna("").astype(str)
    return data

In [ ]:
df=load_csv()
df["tweet_text"]=df["tweet_text"].apply(clean_tweet)
df=df[df["tweet_text"].str.strip().ne("")].copy()
print("Rows:",len(df))

## Unigram TF-IDF — top 20 features

In [ ]:
v_uni=TfidfVectorizer(ngram_range=(1,1),max_features=5000)
X_uni=v_uni.fit_transform(df["tweet_text"])
scores=np.asarray(X_uni.sum(axis=0)).ravel(); names=v_uni.get_feature_names_out()
idx=np.argsort(scores)[-20:][::-1]; top_uni=pd.Series(scores[idx],index=names[idx])
display(top_uni.to_frame("tfidf_sum"))
top_uni.sort_values().plot(kind="barh",figsize=(9,7),title="Top 20 Unigram TF-IDF Features")
plt.tight_layout(); plt.show()
top_uni.rename_axis("feature").reset_index(name="tfidf_sum").to_csv("top_unigram_features.csv",index=False)

## Bigram TF-IDF — top 20 features

In [ ]:
v_bi=TfidfVectorizer(ngram_range=(2,2),max_features=5000)
X_bi=v_bi.fit_transform(df["tweet_text"])
scores=np.asarray(X_bi.sum(axis=0)).ravel(); names=v_bi.get_feature_names_out()
idx=np.argsort(scores)[-20:][::-1]; top_bi=pd.Series(scores[idx],index=names[idx])
display(top_bi.to_frame("tfidf_sum"))
top_bi.sort_values().plot(kind="barh",figsize=(9,7),title="Top 20 Bigram TF-IDF Features")
plt.tight_layout(); plt.show()
top_bi.rename_axis("feature").reset_index(name="tfidf_sum").to_csv("top_bigram_features.csv",index=False)